In [1]:
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\shxpat\AppData\Local\Temp\ipykernel_16244\203863611.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdf(pdf_directory):
    """processing all PDF"""
    all_documents=[]
    pdf_dir=Path(pdf_directory) #pdf_directory path is given in Path function 

    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nprocessing:{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file)) #This is a lang chain libarary for loading PDFs
            documents=loader.load()

            for doc in documents: #Addition of some more meta data info 
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error: {e}")
    return all_documents

all_pdf_documents=process_all_pdf("../data")

Found 5 PDF files to process

processing:ssrn-227349.pdf
Loaded 38 pages

processing:ssrn-2432061.pdf
Loaded 33 pages

processing:ssrn-3406068.pdf
Loaded 23 pages

processing:ssrn-4560455.pdf
Loaded 21 pages

processing:ssrn-6554918.pdf
Loaded 16 pages


In [3]:
len(all_pdf_documents) # Each page is consider a differnt pdf here it is 33+23+21+16+38

131

In [4]:

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")

    return split_docs

In [12]:
chunks=split_documents(all_pdf_documents,chunk_size=400)

Split 131 documents into 1156 chunks

Example chunk:
Content: NBER WORKING PAPER SERIES
PROFITABILITY OF MOMENTUM
STRATEGIES: AN EVALUATION
OF ALTERNATIVE EXPLANATIONS
Narasimhan Jegadeesh
Sheridan Titman
Working Paper 7159
http://www.nber.org/papers/w7 159
NATI...
Metadata:{'producer': 'CVISION Technologies', 'creator': 'CVISION Technologies', 'creationdate': 'D:19990607103905', 'moddate': '2004-06-17T15:53:41-04:00', 'source': '..\\data\\pdf_doc_data\\ssrn-227349.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1', 'source_file': 'ssrn-227349.pdf', 'file_type': 'pdf'}


In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [14]:
class EmbeddingManager:
    def __init__(self,model_name: str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
        
    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.Embedding dim:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            raise
    
    def generate_embeddings(self,text: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts ---")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape:{embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model_get_sentence_embedding_dimension()



In [15]:
embeddingmanager=EmbeddingManager()
embeddingmanager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2273.19it/s]


Model loaded successfully.Embedding dim:384


C:\Users\shxpat\AppData\Local\Temp\ipykernel_16244\3508974203.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully.Embedding dim:{self.model.get_sentence_embedding_dimension()}")


In [16]:
class VectorStore():
    def __init__(self,collection_name: str="pdf_documents",persist_directory: str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.clint=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata=({'discription':"PDF document embedding for RAG"})
            )
            print(f"vector store initialized Collection:{self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            

        except Exception as e:
            print(f"Error")
            raise
    
    def add_documents(self,documents: List[Any],embeddings:np.array):
        if len(documents) !=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding{len(documents)}")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)}")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print("Error")
            raise


In [17]:
vectorstore=VectorStore()
vectorstore

vector store initialized Collection:pdf_documents
Existing documents in collection: 994


In [18]:
texts=[doc.page_content for doc in chunks]

#generate the embedding 
embedding=embeddingmanager.generate_embeddings(texts)

#store in vector
vectorstore.add_documents(chunks,embedding)


Generating embeddings for 1156 texts ---


Batches: 100%|██████████| 37/37 [00:37<00:00,  1.02s/it]


Generated embeddings with shape:(1156, 384)
Adding1156
Successfully added 1156
Total documents in collection: 2150


In [19]:
class RAGRetriever:

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        print(f"Retrieve Query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved: {len(retrieved_docs)}")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Retrieval error: {e}")
            return []

In [20]:
rag_retriever=RAGRetriever(vectorstore,embeddingmanager)

In [21]:
rag_retriever

In [24]:
rag_retriever.retrieve("What is  MOMENTUM STRATEGIES?")

Retrieve Query: What is  MOMENTUM STRATEGIES?
Top K: 5, Score threshold: 0.0
Generating embeddings for 1156 texts ---


Batches: 100%|██████████| 37/37 [01:06<00:00,  1.80s/it]


Generated embeddings with shape:(1156, 384)
Retrieved: 5


[{'id': 'doc_f0a99524_0',
  'content': 'NBER WORKING PAPER SERIES\nPROFITABILITY OF MOMENTUM\nSTRATEGIES: AN EVALUATION\nOF ALTERNATIVE EXPLANATIONS\nNarasimhan Jegadeesh\nSheridan Titman\nWorking Paper 7159\nhttp://www.nber.org/papers/w7 159\nNATIONAL BUREAU OF ECONOMIC RESEARCH\n1050 Massachusetts Avenue\nCambridge, MA 02138\nJune 1999\nThis paper has benefited from the excellent research assistance of Fei Zou and helpful comments from the',
  'metadata': {'moddate': '2004-06-17T15:53:41-04:00',
   'creationdate': 'D:19990607103905',
   'source': '..\\data\\pdf_doc_data\\ssrn-227349.pdf',
   'doc_index': 0,
   'producer': 'CVISION Technologies',
   'total_pages': 38,
   'creator': 'CVISION Technologies',
   'file_type': 'pdf',
   'page': 0,
   'source_file': 'ssrn-227349.pdf',
   'page_label': '1',
   'content_length': 393},
  'similarity_score': 1.0,
  'distance': 0.0,
  'rank': 1},
 {'id': 'doc_f256f115_4',
  'content': 'Profitability of Momentum Strategies:\nAn Evaluation of Alter